# 04 · Je eigen virtuele sensor: ESP32 + DHT22 in Wokwi

Tot slot bouw je zelf een IoT-sensor — in de browser, zonder hardware. **Wokwi** (<https://wokwi.com>) simuleert een **ESP32** (de meest gebruikte wifi-microcontroller, ± € 8) met een **DHT22**-temperatuur-/vochtsensor. De simulator heeft een virtueel wifi-netwerk met echte internettoegang, dus je ESP32 publiceert **echt** naar `broker.hivemq.com`, en de hele klas ziet jouw "stal" in notebook 02.

Wat je hier leert: hoe de code in zo'n toestel eruitziet (wifi verbinden → sensor lezen → JSON maken → MQTT publiceren → slapen), en hoe weinig dat is.

## 1. Project aanmaken (± 5 minuten)

1. Ga naar <https://wokwi.com> en klik **+ New Project** (rechtsboven). Je kunt zonder account simuleren; om je project te **bewaren** heb je een (gratis) account nodig.
2. Kies **MicroPython on ESP32** (sectie *MicroPython*).
3. Je ziet twee tabbladen: `main.py` (code) en `diagram.json` (de schakeling). Vervang de inhoud van **beide** door de bestanden hieronder (ze staan ook in de map `wokwi/` van deze repository).
4. Pas in `main.py` `KLASCODE` en `AFDELING` aan.
5. Klik op de groene **▶ Start**-knop. In de seriële monitor onderaan zie je: wifi verbonden → MQTT verbonden → elke 5 s een uplink.
6. Klik in het schema op de **DHT22** en schuif de temperatuur boven de 30 °C: de volgende uplink krijgt `"interrupt": 1` — het alarm.

> **Openbaar netwerk.** Wokwi stuurt het verkeer van je virtuele ESP32 via een publieke gateway; alles wat je verstuurt is zichtbaar voor Wokwi. Prima voor dit practicum, nooit voor echte gegevens.

## 2. De schakeling: `diagram.json`

Drie draden: voeding (3V3), massa (GND) en de datalijn van de DHT22 naar **GPIO 15**. De DHT22 gebruikt een *1-wire*-achtig protocol over die ene datalijn — een voorbeeld van een draadgebonden sensorbus, net als Modbus RS-485 maar dan voor één sensor op 20 cm.

In [ ]:
# Inhoud van wokwi/diagram.json (ter info — plak dit in het tabblad diagram.json van Wokwi)
print(open('../wokwi/diagram.json').read())

## 3. De code: `main.py`

Lees de code van boven naar beneden; de nummers 1–3 in de commentaar zijn de drie stappen die *elk* IoT-toestel zet. Vergelijk met de Dragino uit notebook 01: daar zit precies dezelfde logica in firmware, en de `AT+…`-commando's zijn niets anders dan de instellingen bovenaan dit script.

In [ ]:
# Inhoud van wokwi/main.py (ter info — plak dit in het tabblad main.py van Wokwi)
print(open('../wokwi/main.py').read())

## 4. Meekijken vanuit dit notebook

Start de simulatie in Wokwi en voer dan deze cel uit: je ziet je eigen virtuele sensor binnenkomen. (Dezelfde instellingen als notebook 02.)

In [ ]:
import os, time, json
import paho.mqtt.client as mqtt

BROKER   = os.environ.get("IOT_BROKER", "broker.hivemq.com")
POORT    = int(os.environ.get("IOT_POORT", "1883"))
KLASCODE = os.environ.get("IOT_KLASCODE", "i700301-2026-xyz")
TOPIC    = f"ugent/{KLASCODE}/stal/+"                # '+' moet een volledig niveau zijn: 'wokwi-+' mag niet

def on_message(client, userdata, msg):
    if not msg.topic.split("/")[-1].startswith("wokwi-"):   # enkel de Wokwi-sensoren tonen
        return
    b = json.loads(msg.payload)
    print(f"[{time.strftime('%H:%M:%S')}] {msg.topic.split('/')[-1]:>16}: {b['temperature']:5.1f} °C  {b['humidity']:5.1f} %  "
          f"bat {b['battery']} V  {'ALARM' if b.get('interrupt') else ''}")

c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
c.on_message = on_message
c.connect(BROKER, POORT, keepalive=60)
c.subscribe(TOPIC)
c.loop_start(); time.sleep(int(os.environ.get("IOT_LUISTER", "30"))); c.loop_stop(); c.disconnect()
print("klaar")

## 5. Opdrachten

1. **Zendinterval en batterij.** Zet `INTERVAL` op 600 zoals een echt toestel. Hoe lang moet je wachten op je eerste meting? Waarom kiezen fabrikanten toch voor zulke lange intervallen? (Dragino-handleiding §4: batterijduur per interval.)
2. **Deep sleep.** Een echte ESP32 op batterij slaapt tussen twee metingen (`machine.deepsleep(ms)`) en start dan volledig opnieuw. Zoek op wat dat betekent voor de wifi- en MQTT-verbinding, en vergelijk met LoRaWAN, waar "verbinden" bijna niets kost.
3. **Een tweede grootheid.** Voeg in Wokwi een tweede sensor toe (bv. een `wokwi-photoresistor-sensor` voor licht of een tweede DHT22) en neem de waarde op in het JSON-bericht. Kan notebook 02 je bericht nog lezen? Wat zegt dat over het belang van een afgesproken berichtformaat?
4. **Van simulatie naar aankoop.** Je wilt dit écht bouwen voor één afdeling, met wifi op het erf. Stel een boodschappenlijst op bij Antratek (ESP32-ontwikkelbord, DHT22 of beter SHT31-module, behuizing, voeding). Vergelijk de prijs en de batterijduur met de Dragino S31-LB. Wanneer kies je welke?

<details>
<summary><b>Oplossing tonen</b></summary>

1. Bij `INTERVAL = 600` wacht je 10 min op de eerste meting. De batterijduur schaalt ongeveer lineair met het interval: zenden (en bij wifi: verbinden) is de grootste verbruiker; Dragino §4 geeft de tabel per interval.
2. Na `machine.deepsleep()` herstart de ESP32 volledig: wifi opnieuw associëren (1–4 s, 100–200 mA) en MQTT opnieuw verbinden. Dat maakt wifi-sensoren op batterij hooguit maanden houdbaar. Bij LoRaWAN blijft de sessie (sleutels, teller) bewaard; een uplink kost enkele tientallen ms zendtijd, vandaar jaren.
3. Zolang je de bestaande sleutels (`temperature`, `humidity`, `time`, …) behoudt en enkel een sleutel toevoegt, leest notebook 02 het bericht nog. Verander je sleutelnamen of de structuur, dan breekt de ontvanger: een **afgesproken berichtformaat** (schema) is de basis van elk IoT-systeem, precies wat de Dragino-handleiding in §2.2 vastlegt.
4. ESP32-ontwikkelbord ± € 10, SHT31-module ± € 10–15 (DHT22 ± € 5), behuizing IP65 ± € 10, USB-voeding ± € 8: ± € 40, maar **netvoeding nodig** (wifi op batterij werkt maanden, geen jaren) en zelf programmeren en onderhouden. Dragino S31-LB € 53 + gateway, batterij tot 10 jaar, kant-en-klaar, maar LoRaWAN-infrastructuur nodig. Zelfbouw wint bij één sensor met stopcontact en wifi in bereik; Dragino wint zodra je meerdere sensoren zonder stroom en zonder onderhoud wilt.

</details>